# Joint Reward Target Scope Guard v1

This notebook audits whether the latest joint reward/checkpoint target table can be used as training data without leaking OOS or walk-forward evaluation labels. It is a guard before BC/PPO, not a performance promotion card.

## Experiment Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`
- **Direct comparison baseline**: `joint_reward_checkpoint_recipe_v1`
- **Role**: `diagnose`
- **Changed layer**: `model_selection`, `selection`, `sampling`
- **Target window**: `wf_2022_train`, `wf_2023_2024_train` train/eval boundaries used by the existing fold-native teacher-router target card.
- **Accepted trade-off**: block a seemingly useful target recipe if it is global/OOS scoped and touches eval dates.
- **Hypothesis**: the joint reward recipe is useful as a loss design, but its current `strict_oos`/global target rows are not safe training labels; the next trainable branch must regenerate fold-local joint targets.
- **Failure mode**: if the current target table is not fold-local, do not start PPO or BC from it; only use it to specify terms.
- **Comparison-table row key**: `joint_reward_target_scope_guard_v1`

## Historical Context

- Closest success: `state_objective_fold_native_teacher_router_v1` produced fold-local targets that passed leakage scope and label distribution gates.
- Closest failure: `joint_reward_checkpoint_recipe_v1` produced strong joint loss terms but selected a global `strict_oos` teacher path and still failed window gates.
- Why this is not a duplicate: this notebook does not rescore candidate paths; it blocks direct training on globally scoped target rows before they can overfit the next PPO/BC branch.

## External Basis

- Qlib workflow separation: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL environment/reward/action separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Backtest overfitting caution: https://doi.org/10.21314/JCF.2016.322
- Sparse/hard sample weighting references for the eventual fold-local branch: Prioritized Experience Replay https://arxiv.org/abs/1511.05952 and Focal Loss https://arxiv.org/abs/1708.02002


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / 'src/soxl_quant').exists()), cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_scope,
    summarize_state_objective_target_scope_audit,
)

CARD = ROOT / 'runtime/model_selection/card/joint_reward_target_scope_guard_v1'
JOINT_RECIPE = ROOT / 'runtime/model_selection/card/joint_reward_checkpoint_recipe_v1'
FOLD_NATIVE = ROOT / 'runtime/model_selection/card/state_objective_fold_native_teacher_router_v1'
CARD.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text())

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n')

folds = [
    {
        'name': 'wf_2022_train',
        'train_start': '2020-01-02',
        'train_end': '2021-12-31',
        'eval_start': '2022-01-03',
        'eval_end': '2022-12-30',
    },
    {
        'name': 'wf_2023_2024_train',
        'train_start': '2020-01-02',
        'train_end': '2022-12-30',
        'eval_start': '2023-01-03',
        'eval_end': '2024-12-31',
    },
]


## Audit Current Joint Recipe Targets

The joint recipe target table contains diagnostic windows such as `strict_oos`. If those rows touch eval periods or have no fold-local scope, they must not be treated as train labels.

In [2]:
joint_selection = read_json(JOINT_RECIPE / 'selection.json')
joint_targets = pd.read_csv(JOINT_RECIPE / 'training_targets.csv')
latest_audit = audit_state_objective_target_scope(joint_targets, folds=folds)
latest_summary = summarize_state_objective_target_scope_audit(latest_audit)

latest_audit.to_csv(CARD / 'joint_recipe_scope_audit.csv', index=False)
write_json(CARD / 'joint_recipe_scope_summary.json', latest_summary)

print(json.dumps(latest_summary, indent=2))
latest_audit[[
    'window', 'target_scope', 'target_fold', 'fold', 'rows',
    'train_overlap_rows', 'eval_overlap_rows', 'outside_train_eval_rows',
    'leakage_gate', 'scope_issue',
]]


{
  "decision": "state_objective_targets_leakage_guard_no_training",
  "training_allowed": false,
  "leakage_gate": false,
  "audited_groups": 8,
  "blocked_groups": 8,
  "eval_overlap_rows": 1254,
  "outside_train_eval_rows": 1447,
  "scope_issues": [
    "global_or_unspecified_target_scope"
  ],
  "top_scope_issue": "global_or_unspecified_target_scope",
  "next_mechanism": "regenerate_fold_local_state_objective_targets"
}


,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue
0,strict_oos,,,wf_2022_train,1605,505,251,849,False,global_or_unspecified_target_scope
1,wf_2020,,,wf_2022_train,252,252,0,0,False,global_or_unspecified_target_scope
2,wf_2022,,,wf_2022_train,250,0,250,0,False,global_or_unspecified_target_scope
3,wf_2023_2024,,,wf_2022_train,251,0,0,251,False,global_or_unspecified_target_scope
4,strict_oos,,,wf_2023_2024_train,1605,756,502,347,False,global_or_unspecified_target_scope
5,wf_2020,,,wf_2023_2024_train,252,252,0,0,False,global_or_unspecified_target_scope
6,wf_2022,,,wf_2023_2024_train,250,250,0,0,False,global_or_unspecified_target_scope
7,wf_2023_2024,,,wf_2023_2024_train,251,0,251,0,False,global_or_unspecified_target_scope


## Compare Existing Fold-Native Targets

This comparison proves the guard is not trivially blocking every target table. The existing fold-native teacher-router target table is scoped to train folds and remains eligible as a training source, although it does not yet encode the latest joint reward/checkpoint loss terms.

In [3]:
fold_native_selection = read_json(FOLD_NATIVE / 'selection.json')
fold_native_targets = pd.read_csv(FOLD_NATIVE / 'fold_native_teacher_targets.csv')
fold_native_audit = audit_state_objective_target_scope(fold_native_targets, folds=folds)
fold_native_summary = summarize_state_objective_target_scope_audit(fold_native_audit)

fold_native_audit.to_csv(CARD / 'fold_native_scope_audit.csv', index=False)
write_json(CARD / 'fold_native_scope_summary.json', fold_native_summary)

print(json.dumps(fold_native_summary, indent=2))
fold_native_audit[[
    'window', 'target_scope', 'target_fold', 'fold', 'rows',
    'train_overlap_rows', 'eval_overlap_rows', 'outside_train_eval_rows',
    'leakage_gate', 'scope_issue',
]]


{
  "decision": "state_objective_targets_training_ready",
  "training_allowed": true,
  "leakage_gate": true,
  "audited_groups": 2,
  "blocked_groups": 0,
  "eval_overlap_rows": 0,
  "outside_train_eval_rows": 0,
  "scope_issues": [],
  "top_scope_issue": "",
  "next_mechanism": "run_fold_local_state_objective_training_or_bc"
}


,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue
0,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,505,505,0,0,True,
1,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,


## Decision

The current joint recipe is valid as a target design and checkpoint-loss handoff, but not as direct training labels. The next trainable branch should regenerate the same joint terms inside each fold-local train window, then run weighted BC/PPO and retain checkpoints by `checkpoint_objective_loss`.

In [4]:
latest_training_allowed = bool(latest_summary['training_allowed'])
fold_native_training_allowed = bool(fold_native_summary['training_allowed'])

if latest_training_allowed:
    decision = 'joint_reward_targets_training_ready'
    next_mechanism = 'run_joint_reward_weighted_bc_or_ppo'
else:
    decision = 'joint_reward_targets_scope_blocked_no_training'
    next_mechanism = 'regenerate_fold_local_joint_reward_targets'

selection = {
    'experiment': 'joint_reward_target_scope_guard_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'joint_reward_checkpoint_recipe_v1',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'selection', 'sampling'],
    'decision': decision,
    'next_mechanism': next_mechanism,
    'promotion_allowed': False,
    'training_allowed_from_joint_recipe_targets': latest_training_allowed,
    'fold_native_reference_training_allowed': fold_native_training_allowed,
    'joint_recipe_eval_overlap_rows': int(latest_summary['eval_overlap_rows']),
    'joint_recipe_blocked_groups': int(latest_summary['blocked_groups']),
    'joint_recipe_scope_issues': latest_summary['scope_issues'],
    'fold_native_blocked_groups': int(fold_native_summary['blocked_groups']),
    'required_next_terms': joint_selection['required_terms'],
    'promotion_blocker': (
        'Latest joint recipe target table is globally scoped and touches evaluation dates; '
        'regenerate fold-local joint targets before BC/PPO to avoid training on OOS labels.'
    ),
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
        'https://arxiv.org/abs/1511.05952',
        'https://arxiv.org/abs/1708.02002',
    ],
    'acceptance_gate_for_next_run': {
        'fold_local_target_scope_required': True,
        'eval_overlap_rows_required': 0,
        'checkpoint_retention_metric_required': 'checkpoint_objective_loss',
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
        'fixed_split_required': True,
        'walkforward_required': True,
        'no_runtime_promotion_from_scope_guard': True,
    },
    'sidecars': [
        'joint_recipe_scope_audit.csv',
        'joint_recipe_scope_summary.json',
        'fold_native_scope_audit.csv',
        'fold_native_scope_summary.json',
        'summary.json',
    ],
}

summary = {
    'experiment': 'joint_reward_target_scope_guard_v1',
    'decision': decision,
    'training_allowed_from_joint_recipe_targets': latest_training_allowed,
    'fold_native_reference_training_allowed': fold_native_training_allowed,
    'joint_recipe_eval_overlap_rows': int(latest_summary['eval_overlap_rows']),
    'joint_recipe_blocked_groups': int(latest_summary['blocked_groups']),
    'next_mechanism': next_mechanism,
    'conclusion': [
        f"Joint recipe targets are not train-ready: blocked_groups={latest_summary['blocked_groups']}, eval_overlap_rows={latest_summary['eval_overlap_rows']}.",
        f"Fold-native reference targets remain train-ready: training_allowed={fold_native_training_allowed}.",
        'Next branch should regenerate fold-local joint reward targets before weighted BC/PPO; do not train directly from strict_oos/global recipe rows.',
    ],
}

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(summary, indent=2))


{
  "experiment": "joint_reward_target_scope_guard_v1",
  "decision": "joint_reward_targets_scope_blocked_no_training",
  "training_allowed_from_joint_recipe_targets": false,
  "fold_native_reference_training_allowed": true,
  "joint_recipe_eval_overlap_rows": 1254,
  "joint_recipe_blocked_groups": 8,
  "next_mechanism": "regenerate_fold_local_joint_reward_targets",
  "conclusion": [
    "Joint recipe targets are not train-ready: blocked_groups=8, eval_overlap_rows=1254.",
    "Fold-native reference targets remain train-ready: training_allowed=True.",
    "Next branch should regenerate fold-local joint reward targets before weighted BC/PPO; do not train directly from strict_oos/global recipe rows."
  ]
}
